In [ ]:
import numpy as np
import seaborn as sns
from colorspacious import cspace_convert, deltaE
from scipy.optimize import minimize
from scipy.special import logsumexp

REFERENCE_COLORS = np.array([[1.0, 1.0, 1.0], [0.5, 0.5, 0.5], [0.0, 0.0, 0.0]])

NORMAL_WEIGHT = 1.00
CVD_WEIGHT = 0.027
BW_WEIGHT = 0.01


def distance_matrix(srgb: np.ndarray) -> np.ndarray:
    return deltaE(
        srgb[:, None, :],
        srgb[None, :, :],
        input_space="sRGB1",
        uniform_space="CAM02-SCD",
    )


def triu_values(matrix: np.ndarray) -> np.ndarray:
    n = matrix.shape[0]
    return matrix[np.triu_indices(n, k=1)]


def simulate_deuteranomaly(srgb: np.ndarray, severity: int = 50) -> np.ndarray:
    cvd_space = dict(name="sRGB1+CVD", cvd_type="deuteranomaly", severity=severity)
    return cspace_convert(srgb, cvd_space, "sRGB1")


def to_greyscale(srgb: np.ndarray) -> np.ndarray:
    jch = cspace_convert(srgb, "sRGB1", "JCh")
    jch[:, 1:] = 0.0
    return cspace_convert(jch, "JCh", "sRGB1")


def intensity(srgb: np.ndarray) -> np.ndarray:
    # Use CIELAB chroma as a proxy for intesity
    cielab = cspace_convert(srgb.reshape(-1, 3), "sRGB1", "CIELab")
    return np.sqrt(cielab[:, 1] ** 2 + cielab[:, 2] ** 2)


def nearest_neighbor_order(srgb: np.ndarray) -> np.ndarray:
    """Return an index array that orders colors to minimise total neighbour distance.

    Tries every starting color and keeps the greedy nearest-neighbor path with
    the lowest total cost (O(n^3), practical for small palettes).
    """
    dists = distance_matrix(srgb)
    n = len(srgb)
    best_order, best_cost = None, np.inf
    for start in range(n):
        visited = [False] * n
        order = [start]
        visited[start] = True
        cost = 0.0
        for _ in range(n - 1):
            current = order[-1]
            nearest = min(
                (j for j in range(n) if not visited[j]),
                key=lambda j: dists[current, j],
            )
            cost += dists[current, nearest]
            order.append(nearest)
            visited[nearest] = True
        if cost < best_cost:
            best_cost, best_order = cost, order
    return np.array(best_order)


def score_perceptual_distance(vec_srgb: np.ndarray) -> float:
    srgb = vec_srgb.reshape(-1, 3)
    full_set = np.vstack([srgb, REFERENCE_COLORS])

    cdist_vec = triu_values(distance_matrix(full_set))
    cvd_cdist_vec = triu_values(distance_matrix(simulate_deuteranomaly(full_set)))
    bw_cdist_vec = triu_values(distance_matrix(to_greyscale(full_set)))

    intensity_vec = intensity(srgb)
    # Hinge loss to discourage high intensity colours
    hinge_high = np.maximum(
        0, intensity_vec - 85
    ).sum()  # Adjust the threshold as needed
    hinge_low = np.maximum(
        0, 20 - intensity_vec
    ).sum()  # Adjust the threshold as needed
    hinge = hinge_high + hinge_low

    return (
        logsumexp(-NORMAL_WEIGHT * cdist_vec)
        + logsumexp(-CVD_WEIGHT * cvd_cdist_vec)
        + logsumexp(-BW_WEIGHT * bw_cdist_vec)
        + hinge
    )  # type: ignore


colors = sns.color_palette("tab10", 10)
initial_srgb = np.array(colors)
rng = np.random.default_rng(28)
# initial_srgb = rng.random(initial_srgb.shape)
vec_srgb = initial_srgb.flatten()

original_score = score_perceptual_distance(vec_srgb)

solution = minimize(
    score_perceptual_distance,
    vec_srgb,
    bounds=[(0, 1)] * len(vec_srgb),
    tol=1e-6,
)

optimized_score = score_perceptual_distance(solution.x)
print(f"{original_score:.4f} -> {optimized_score:.4f}")

In [ ]:
print(
    "Original Min Perceptual Distance ",
    np.min(triu_values(distance_matrix(initial_srgb))),
)
print(
    "Optimized Min Perceptual Distance",
    np.min(triu_values(distance_matrix(solution.x.reshape(-1, 3)))),
)

In [ ]:
import matplotlib.pyplot as plt


def palette_views(srgb: np.ndarray) -> list[np.ndarray]:
    return [
        srgb,
        to_greyscale(srgb),
        np.clip(simulate_deuteranomaly(srgb), 0.0, 1.0),
    ]


original_srgb = np.array(initial_srgb).reshape(-1, 3)
optimized_srgb = solution.x.reshape(-1, 3)
optimized_srgb = optimized_srgb[nearest_neighbor_order(optimized_srgb)]

rows = palette_views(original_srgb) + palette_views(optimized_srgb)
labels = [
    "Original",
    "Original (B/W)",
    "Original (Deuteranomaly)",
    "Optimized",
    "Optimized (B/W)",
    "Optimized (Deuteranomaly)",
]

plot_data = np.stack(rows, axis=0)

fig, ax = plt.subplots(figsize=(10, 4.2))
ax.imshow(plot_data, aspect="auto")

ax.hlines(
    np.arange(-0.5, len(rows), 1),
    -0.5,
    len(original_srgb) - 0.5,
    color="white",
    linewidth=2,
)
ax.vlines(
    np.arange(-0.5, len(original_srgb), 1),
    -0.5,
    len(rows) - 0.5,
    color="white",
    linewidth=2,
)

ax.set_yticks(np.arange(len(labels)))
ax.set_yticklabels(labels)
ax.set_xticks(np.arange(len(original_srgb)))
ax.set_xticklabels([f"Color {i + 1}" for i in range(len(original_srgb))])
ax.set_title("Palette Comparison: Normal, B/W, and Deuteranomaly")
plt.tight_layout()
plt.show()

In [ ]:
for color in optimized_srgb:
    print(color)

In [ ]:
intensities_original = intensity(original_srgb)
intensities_optimized = intensity(optimized_srgb)

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(original_srgb))
width = 0.35
ax.bar(x - width / 2, intensities_original, width, label="Original", color="blue")
ax.bar(x + width / 2, intensities_optimized, width, label="Optimized", color="orange")
ax.set_xticks(x)
ax.set_xticklabels([f"Color {i + 1}" for i in range(len(original_srgb))])
ax.set_ylabel("Intensity (CIE Chroma)")
ax.set_title("Intensity Comparison of Original and Optimized Palettes")
ax.legend()

In [ ]:
# Sort optimized color map by intensity
sorted_indices = np.argsort(-intensities_optimized)
sorted_optimized_srgb = optimized_srgb[sorted_indices]

# Display sorted optimized palette
fig, ax = plt.subplots(figsize=(8, 2))
ax.imshow(sorted_optimized_srgb[np.newaxis, :, :], aspect="auto")
ax.set_yticks([])
ax.set_xticks(np.arange(len(sorted_optimized_srgb)))
ax.set_xticklabels([f"Color {i + 1}" for i in range(len(sorted_optimized_srgb))])
ax.set_title("Optimized Palette Sorted by Intensity")

In [ ]:
sorted_indices